> ## 🟢 VERSION 2 — what changed vs the original notebook
>
> Same **kit** (the `pipeline/` package + `bruise_colab_gpu_full.zip`) and same models,
> sweep, accuracy and outputs. Three steps are swapped to match
> `bruise_colab_train_per_model_batch.ipynb`:
>
> 1. **Dataloader source** — images are cv2-resized to 640 **once** into a PNG cache,
>    then read from that cache (instead of resizing the native 4022×6024 image on every
>    read).
> 2. **Pixel scale** — the dataloader emits a neutral **`/255`** tensor; each model then
>    rescales in `model_input()` (ImageNet norm for SegFormer, `/255` kept for YOLO).
>    Algebraically identical to v1's ImageNet-loader-then-de-normalise-YOLO path.
> 3. **Timing helper** — the benchmark thresholds the **raw logit** (`z >= cut`, no
>    sigmoid), exactly like per_model_batch's `benchmark_speed`.
>
> Everything runs over **all 185 test images**. The original notebook is untouched.

---

# Bruise Segmentation — Threshold Sweep, Inference & Evaluation

Runs our **5 trained models** end to end, entirely in PyTorch:

1. **Fit** each model's operating point with a threshold sweep on the **134-image val set**.
2. **Time** each model — 640 in → 640 out, on the GPU, over all **185 test images**.
3. **Score** each model — Dice / IoU / complete-miss on those same 185 test images.

---

## One path

| stage | what happens | same for all 5? |
|---|---|---|
| read + resize | `BruiseDataset` — cv2 stretch-resize to 640 | ✅ identical |
| pixel scale | ImageNet norm (SegFormer) · `/255` (YOLO) | ⚠️ **per-model, and it has to be** |
| forward | raw PyTorch `nn.Module(x)` | ✅ identical |
| logits → 640 | bilinear upsample | ✅ identical |
| logits → bruise logit | 1ch: as-is · 2ch: `z₁−z₀` | ✅ identical |
| → mask | `sigmoid(z) ≥ threshold` | ✅ identical |
| threshold | 1-D cut sweep on **val**, in this notebook | ✅ identical |

No letterbox. No `.predict()`. No `cv2.resize` on any model output. No temperature. One
dataloader, one resize, one disk read.

### ⚠️ The one thing that is *not* uniform — and why

An earlier draft of this notebook forced **every** model onto ImageNet normalisation, on
the theory that identical input makes the comparison fair. Measured on real val images,
that theory is wrong:

| YOLO input recipe | Dice @ argmax | **best Dice at *any* threshold** |
|---|---|---|
| ImageNet norm | 0.3092 | **0.4791** |
| `/255` (what it was trained on) | 0.7507 | **0.7940** |

Ultralytics trains on plain `/255`; its BatchNorms carry frozen running statistics for
*that* distribution and cannot adapt. Feeding ImageNet-normalised pixels makes YOLO
under-fire by **4×** (predicting 1.2% positive pixels against a 4.7% GT rate), and **no
threshold recovers it** — 0.479 is the ceiling.

So the pixel scale is treated as **a property of the weights, like the weights
themselves** — not as a pipeline choice. `model_input()` undoes the dataloader's
normalisation for YOLO (`x*STD + MEAN`, exact to ~6e-8), which recovers the exact `/255`
image it was trained on **without** a second dataloader, a second disk read, or a
different resize. Geometry stays byte-identical across all 5 models.

### Why the two families still collapse into one function

SegFormer emits **1** channel; YOLO emits **2**. But for a 2-class head,

$$P(\text{bruise}) = \frac{e^{z_1}}{e^{z_0}+e^{z_1}} = \sigma(z_1 - z_0)$$

2-class softmax **is** a sigmoid of the logit difference. So `z₁−z₀` is YOLO's single
bruise logit, exactly analogous to SegFormer's, and both go through the same
`sigmoid → threshold`. The code branches on **channel count** — a structural fact —
never on a model's name.

(Ultralytics' `argmax` is this rule frozen at 0.5: `argmax==1` ⟺ `z₁>z₀` ⟺ `σ(z₁−z₀)>0.5`.
Doing it ourselves costs nothing and gives us the threshold control their API hides.)

### 🔬 Why there is no temperature any more

The old sweep searched a 2-D grid of `(temperature, threshold)` — 8 × 19 = 152 points.
That grid was **mathematically redundant**:

$$\sigma(z/T) \ge \text{thr} \iff z \ge T\cdot\text{logit}(\text{thr})$$

For a **hard mask**, the decision depends only on the *product* `c = T·logit(thr)`, never
on `T` and `thr` separately. Verified on the real model — two different grid points with
the same cut return **bit-identical** Dice:

| T | thr | `c = T·logit(thr)` | measured mean Dice |
|---|---|---|---|
| 2.0 | 0.25 | **−2.197225** | **0.40995198** |
| 1.0 | 0.10 | **−2.197225** | **0.40995198** |

Of 152 old grid points only **116** were distinct outcomes. And temperature's only real
effect — widening the reachable cut range to `[−23.6, 23.6]` versus `[−2.94, 2.94]` at
`T=1` — bought nothing, since the optima sit well inside what `T=1` already reaches.

So the sweep below is **1-D**: it sweeps the cut `c` directly on the raw logit — the space
the decision actually lives in — and reports the equivalent threshold as `σ(c)`.

## 1. Mount Google Drive

> ⚠️ **This notebook needs the rebuilt package** (~1.2 GB), which adds the 134 val
> images the sweep fits on. The old ~868 MB zip shipped **test images only** and
> aliased `train_manifest` to the test manifest — a sweep run against it would have
> silently fitted the threshold on the test set. Rebuild with
> `python scripts/32_build_colab_gpu_package.py` and re-upload before running this.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/bruise_segmentation_gpu'
ZIP_PATH  = f'{DRIVE_DIR}/bruise_colab_gpu_full.zip'

import os
assert os.path.exists(ZIP_PATH), f'{ZIP_PATH} not found — upload the zip to {DRIVE_DIR}/ first.'
size_mb = os.path.getsize(ZIP_PATH) / 1e6
print('Found package:', ZIP_PATH, f'({size_mb:.0f} MB)')
assert size_mb > 1000, (
    f'This zip is {size_mb:.0f} MB — that looks like the OLD test-only package. The val '
    'split (needed to fit thresholds without touching test) makes it ~1.2 GB. '
    'Rebuild with scripts/32_build_colab_gpu_package.py and re-upload.')

## 2. Check we actually have a GPU

Timing numbers are meaningless without one. This stops the notebook immediately rather
than letting you collect nonsense. **Runtime → Change runtime type → A100 GPU.**

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > A100 GPU, then re-run.'
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(0))

## 3. Copy the zip to local disk and unzip

We copy off Drive first — reading images over the Drive mount adds unpredictable network
latency to preprocessing.

In [ ]:
import shutil, time
LOCAL_DIR = '/content/bruise_gpu'

t0 = time.time()
shutil.copy(ZIP_PATH, '/content/pkg.zip')
print(f'Copied in {time.time()-t0:.0f}s')

!rm -rf {LOCAL_DIR}
!mkdir -p {LOCAL_DIR}
t0 = time.time()
!unzip -q /content/pkg.zip -d {LOCAL_DIR}
print(f'Unzipped in {time.time()-t0:.0f}s')

## 4. Install the libraries

Colab already ships torch + CUDA; reinstalling torch would break the GPU build.

In [ ]:
%cd {LOCAL_DIR}
!pip install -q transformers ultralytics albumentations opencv-python-headless

## 5. Imports and configuration

> ### 🐛 The `cv2` / `ultralytics` mask bug — and why import order does *not* fix it
>
> Older versions of this notebook imported `cv2` before `ultralytics` "on purpose",
> claiming that avoided `cv2.imread(..., IMREAD_GRAYSCALE)` returning `(H,W,1)` instead
> of `(H,W)`. **That workaround is a myth.** Measured on the real masks:
>
> | | `imread` GRAYSCALE result |
> |---|---|
> | cv2 alone, no ultralytics | `(4022, 6024)` ✅ |
> | ultralytics imported **first** | `(4022, 6024, 1)` ❌ |
> | cv2 imported **first** | `(4022, 6024, 1)` ❌ |
>
> Ultralytics monkey-patches `cv2.imread` at import time. Once it is imported *anywhere*
> in the process, the patch is live — order is irrelevant.
>
> **Why this matters more than it looks.** That trailing axis survives albumentations and
> `ToTensorV2`, so `y` arrives as `[B,1,640,640,1]`. Nothing crashes. Instead
> `dice_np(pred[640,640], gt[640,640,1])` **broadcasts** to `[640,640,640]` and returns
> nonsense — a pixel-perfect prediction scores **Dice 63.9 instead of 1.0**.
>
> Fixed at the source: `pipeline/data.py` now squeezes the trailing axis after `imread`
> (a no-op when cv2 is unpatched). The assert in §6 verifies the fix held — it is the
> difference between a real number and a fake one, so it is checked, not assumed.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import torch, torch.nn.functional as F
import copy, sys, json, time
from pathlib import Path
from torch.utils.data import DataLoader

sys.path.insert(0, LOCAL_DIR)
from pipeline.io_utils import load_yaml
from pipeline.data import BruiseDataset, load_fixed_test
from pipeline.models import load_segformer_model, count_params
from pipeline.metrics import compute_image_row

paths = load_yaml('configs/paths.yaml')
cfg   = load_yaml('configs/common_train.yaml')
IMG_H = IMG_W = 640                      # every model runs and is scored at 640x640
assert (cfg['img_h'], cfg['img_w']) == (IMG_H, IMG_W)

torch.backends.cudnn.benchmark = True    # autotune conv kernels; input shape is fixed
print('Config loaded. Grid:', IMG_H, 'x', IMG_W)

# Import ultralytics HERE, up front, rather than next to the YOLO loading code. Not for
# ordering reasons -- the header above shows order is irrelevant -- but so that every
# dataloader in this notebook runs under the SAME cv2 patch state. Importing it later
# (next to the model loading) meant the test set was read with an unpatched cv2 and the
# val set with a patched one, so the two GT tensors came out different shapes depending
# on which cells you had run. Same state everywhere is the actual fix; data.py's squeeze
# then makes that state harmless.
import ultralytics
from ultralytics import YOLO
print('ultralytics', ultralytics.__version__, '| cv2', cv2.__version__)

## 6. Load val + test — and prove they don't overlap

`BruiseDataset` is the same loader used in training, and here it is the **only** thing
that reads pixels. Per image it reads the native photo (4022×6024) and its mask,
cv2-resizes both to 640×640 (image bilinear, mask nearest — nearest keeps the mask
strictly 0/1), ImageNet-normalises the image, and returns tensors.

**Every model gets this exact tensor and this exact geometry.** The only later difference
is that `model_input()` (§9) rescales the pixels for YOLO — no second read, no second
resize.

> ### 🛑 The leak guard below is not decoration
> The whole reason the val split is shipped is so the threshold can be fitted somewhere
> other than the set it's scored on. The **previous** package aliased `train_manifest`
> to `fixed_test_manifest` — had we swept against that, the threshold would have been
> fitted on test and then used to report test scores. That wouldn't crash; it would just
> quietly hand back inflated numbers, which is far worse. The asserts check the
> manifests are different files **and** that no image *and no subject* is in both.

> ### 🛑 So is the mask-shape guard
> With ultralytics imported, `cv2.imread` is patched and GT can silently arrive as
> `(640,640,1)`. That doesn't crash either — it broadcasts, and a *pixel-perfect*
> prediction scores **Dice 63.9**. Both splits are checked.

In [ ]:
val_df  = load_fixed_test(paths['val_manifest'])          # 134 images — thresholds fitted here
test_df = load_fixed_test(paths['fixed_test_manifest'])   # 185 images — scored here, never fitted

# --- leak guard (unchanged from v1) ---------------------------------------
assert Path(paths['val_manifest']).resolve() != Path(paths['fixed_test_manifest']).resolve(), (
    'val_manifest and fixed_test_manifest are the SAME FILE. Sweeping would fit the '
    'threshold on the test set. Rebuild the package with scripts/32_build_colab_gpu_package.py.')
img_overlap = set(val_df.image_path) & set(test_df.image_path)
sub_overlap = set(val_df.subject)    & set(test_df.subject)
assert not img_overlap, f'{len(img_overlap)} image(s) in BOTH val and test: {sorted(img_overlap)[:3]}'
assert not sub_overlap, f'{len(sub_overlap)} subject(s) in BOTH val and test: {sorted(sub_overlap)[:3]}'
print('Leak guard passed: 0 shared images, 0 shared subjects.')

# === [per_model_batch step 1] DATALOADER SOURCE: build a 640 cache ONCE =====================
# v1 read the native 4022x6024 photo and cv2-resized it to 640 on EVERY __getitem__. Here we
# do that resize a single time, write 640x640 PNGs to local disk, and read those thereafter --
# exactly what per_model_batch does. Image uses INTER_LINEAR (bilinear), mask INTER_NEAREST
# (keeps it strictly 0/1) -- bit-identical geometry to v1's albumentations A.Resize, so no
# accuracy number changes; only the per-read cost drops.
from torch.utils.data import Dataset, DataLoader
CACHE640 = Path(LOCAL_DIR) / 'cache640'

def build_cache(df, split):
    idir = CACHE640 / split / 'images'; mdir = CACHE640 / split / 'masks'
    idir.mkdir(parents=True, exist_ok=True); mdir.mkdir(parents=True, exist_ok=True)
    for _, r in df.iterrows():
        ip = idir / f'{r.stem}.png'; mp = mdir / f'{r.stem}.png'
        if not ip.exists():
            im = cv2.imread(str(r.image_path), cv2.IMREAD_COLOR)
            if im is None:
                raise ValueError(f'Cannot read image: {r.image_path}')
            cv2.imwrite(str(ip), cv2.resize(im, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR))
        if not mp.exists():
            m = cv2.imread(str(r.mask_path), cv2.IMREAD_GRAYSCALE)
            if m is None:
                raise ValueError(f'Cannot read mask: {r.mask_path}')
            if m.ndim == 3:                              # ultralytics' cv2.imread patch adds a trailing axis
                m = m[..., 0]
            b = (m > 0).astype(np.uint8)
            cv2.imwrite(str(mp), cv2.resize(b, (IMG_W, IMG_H), interpolation=cv2.INTER_NEAREST) * 255)

if not (CACHE640 / 'test' / 'images').exists():
    t0 = time.time()
    build_cache(val_df, 'val'); build_cache(test_df, 'test')
    print(f'640 cache built in {time.time()-t0:.0f}s')
else:
    print('640 cache present')


# === [per_model_batch step 2] PIXEL SCALE (loader side): emit a neutral /255 tensor ==========
# per_model_batch's dataloader is model-agnostic: it returns /255, and each model rescales
# later (in model_input, §9). That is the opposite of v1, whose loader ImageNet-normalised and
# then de-normalised YOLO -- algebraically the same thing, but this matches per_model_batch.
class Cache640Dataset(Dataset):
    """Reads the pre-resized 640 cache. Returns (x[3,640,640] in [0,1] /255, y[1,640,640] 0/1, stem)."""
    def __init__(self, df, split):
        self.df = df.reset_index(drop=True); self.split = split
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = cv2.imread(str(CACHE640 / self.split / 'images' / f'{r.stem}.png'), cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0      # <-- /255, nothing else
        x = torch.from_numpy(img.transpose(2, 0, 1))                              # [3,640,640]
        m = cv2.imread(str(CACHE640 / self.split / 'masks' / f'{r.stem}.png'), cv2.IMREAD_GRAYSCALE)
        if m.ndim == 3: m = m[..., 0]
        y = torch.from_numpy((m > 0).astype(np.float32)).unsqueeze(0)             # [1,640,640]
        return x, y, str(r.stem)

val_ds  = Cache640Dataset(val_df,  'val')
test_ds = Cache640Dataset(test_df, 'test')

# --- mask-shape guard (see the cv2/ultralytics note in §5), now on the cache ---------------
for label, ds in (('val', val_ds), ('test', test_ds)):
    _x, _y, *_ = ds[0]
    assert _y.shape == (1, IMG_H, IMG_W), (
        f'{label} GT is {tuple(_y.shape)}, expected (1, {IMG_H}, {IMG_W}).')
    assert _x.shape == (3, IMG_H, IMG_W), (f'{label} image is {tuple(_x.shape)}.')
print(f'Mask-shape guard passed: GT is (1, {IMG_H}, {IMG_W}) for both splits.\n')

val_loader  = DataLoader(val_ds,  batch_size=1, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

x0, y0, *_ = test_ds[0]
print(f'Val  (fit thresholds) : {len(val_df):3d} images, {val_df.subject.nunique()} subjects')
print(f'Test (report scores)  : {len(test_df):3d} images, {test_df.subject.nunique()} subjects')
print(f'Image tensor          : {tuple(x0.shape)}  (640 cache, /255 -- rescaled per-model in model_input)')
print(f'GT mask tensor        : {tuple(y0.shape)}  (0/1)')
print(f'Pixel range           : [{x0.min():.3f}, {x0.max():.3f}]  (should be ~[0, 1])')

## 7. Stage the test images on the GPU, once

The benchmark measures **640 in → 640 out on the GPU**, so the host→device copy must sit
outside the timer. We run the test loader once here and park all 185 preprocessed
tensors in GPU memory (≈0.9 GB), so the timed loop touches nothing but the model.

Val is *not* staged — the sweep isn't timed, so it streams instead and keeps the memory.

In [ ]:
_imgs, _gts, STEMS = [], [], []
for x, y, stems, *_ in test_loader:
    _imgs.append(x.to(DEVICE, non_blocking=True))
    _gts.append(y)                                    # GT stays on CPU; metrics are numpy
    STEMS.extend(stems)

X_TEST = torch.cat(_imgs, dim=0)
GT_640 = torch.cat(_gts, dim=0)
del _imgs, _gts

assert X_TEST.shape == (len(test_df), 3, IMG_H, IMG_W), X_TEST.shape
assert GT_640.shape == (len(test_df), 1, IMG_H, IMG_W), GT_640.shape
assert len(STEMS) == len(test_df)
print(f'Staged on GPU : {tuple(X_TEST.shape)}  ({X_TEST.element_size()*X_TEST.nelement()/1e9:.2f} GB)')
print(f'GT on CPU     : {tuple(GT_640.shape)}')
print(f'Stems         : {len(STEMS)}  (e.g. {STEMS[0]})')

## 8. Load the 5 trained models — all as raw PyTorch `nn.Module`s

Every model becomes a plain `nn.Module` you call as `model(x)`. No wrapper with hidden
preprocessing, no framework-specific inference API.

**No thresholds are read here.** Each model's `threshold` stays `None` until §10 sweeps
it on val. We *do* read the old `threshold_search.csv` into `csv_threshold`, purely so
§10 can print our sweep next to the historical value as a cross-check.

> ### ⚠️ Licensing — the honest version
> Skipping `.predict()` keeps Ultralytics' **inference/postprocessing** stack out of our
> runtime path. But `best.pt` is a *pickled Ultralytics model object*: deserialising it
> still runs `import ultralytics` and still constructs their classes. **This reduces
> AGPL exposure; it does not remove it.** Only a TorchScript/ONNX export drops the
> import — and even that removes the *technical* dependency, not necessarily the
> *legal* one (whether a graph traced from AGPL source is a derivative work is
> contested, and Ultralytics' own position is that models trained with their code are
> AGPL). Not legal advice.

In [ ]:
import contextlib, warnings
from transformers.utils import logging as hf_logging

@contextlib.contextmanager
def quiet_pretrained_load():
    """Silence HF's load report while building a SegFormer.

    build_segformer() starts from an ImageNet *classification* checkpoint, which has no
    segmentation decoder -- so HF correctly reports every decode_head.* param as MISSING
    and the 1000-class classifier.* as UNEXPECTED. load_segformer_model() then loads our
    fine-tuned state_dict over it with strict key matching, so those params are all
    overwritten before use. The report describes an intermediate state, not the loaded
    model. Errors still raise; only warnings are hidden.
    """
    prev = hf_logging.get_verbosity()
    hf_logging.set_verbosity_error()
    hf_logging.disable_progress_bar()
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            yield
    finally:
        hf_logging.set_verbosity(prev)
        hf_logging.enable_progress_bar()


def historical_threshold(run_dir):
    """The old threshold_search.csv's best row, as an equivalent T=1 threshold.

    Reference only -- nothing downstream depends on it. The old YOLO grids swept
    (T, thr); since the mask depends only on c = T*logit(thr), the comparable T=1
    threshold is sigmoid(c). For SegFormer (no temperature column) that is just thr.
    Returns None if the file isn't there.
    """
    csv = Path(run_dir) / 'threshold_search.csv'
    if not csv.exists():
        return None
    row = pd.read_csv(csv).sort_values('mean_dice', ascending=False).iloc[0]
    thr = float(row['threshold'])
    temp = float(row['temperature']) if 'temperature' in row.index else 1.0
    cut = temp * np.log(thr / (1.0 - thr))
    return float(1.0 / (1.0 + np.exp(-cut)))


MODELS = {}

# --- SegFormer: 1-channel head ---
for run, disp, pkey in [
    ('segformer_b2_teacher',   'SegFormer-B2 (teacher)',   'segformer_b2_pretrained'),
    ('segformer_b0_direct',    'SegFormer-B0 (direct)',    'segformer_b0_pretrained'),
    ('segformer_b0_distilled', 'SegFormer-B0 (distilled)', 'segformer_b0_pretrained'),
]:
    with quiet_pretrained_load():
        model, _csv_thr, ckpt = load_segformer_model(run, pkey, paths, DEVICE)
    MODELS[run] = {'display': disp, 'model': model.to(DEVICE).eval(),
                   'input': 'imagenet',                     # MiT encoder pretrained ImageNet-normalised
                   'threshold': None,                       # set by the val sweep in §10
                   'csv_threshold': historical_threshold(f'{LOCAL_DIR}/{run}'),
                   'params_m': count_params(model)[0] / 1e6, 'ckpt': str(ckpt)}

# --- YOLO: 2-channel head. Pull the nn.Module out of the wrapper, drop the wrapper.
# YOLO() is used ONLY to unpickle; the wrapper (and its .predict()) is discarded here.
for run, disp in [('yolo_sem_direct',    'YOLO26n-sem (direct)'),
                  ('yolo_sem_distilled', 'YOLO26n-sem (distilled)')]:
    ckpt = f'{LOCAL_DIR}/{run}/ultralytics_runs/train/weights/best.pt'
    nn_model = copy.deepcopy(YOLO(ckpt).model).to(DEVICE).eval()
    MODELS[run] = {'display': disp, 'model': nn_model,
                   'input': '01',                           # Ultralytics trains on /255, NOT ImageNet norm
                   'threshold': None,                       # set by the val sweep in §10
                   'csv_threshold': historical_threshold(f'{LOCAL_DIR}/{run}'),
                   'params_m': sum(p.numel() for p in nn_model.parameters()) / 1e6,
                   'ckpt': ckpt}

for v in MODELS.values():
    hist = 'n/a' if v['csv_threshold'] is None else f"{v['csv_threshold']:.4f}"
    print(f"{v['display']:26s} {v['params_m']:6.2f}M | threshold: pending sweep "
          f"(historical CSV equiv: {hist})")

## 9. The one inference path

Every number in this notebook comes out of these functions. There is no second path.

### Two per-model properties, both structural

`model_input()` branches on **the pixel scale the weights were trained on**, and
`bruise_logit_640()` branches on **the head's channel count**. Neither branches on a
model's name — both are facts about the checkpoint, in the same category as its weights.

| model | trained pixel scale | raw head output | → bruise logit |
|---|---|---|---|
| SegFormer B2/B0 | ImageNet norm | `[1, 1, 160, 160]` (stride 4) | `out[:, 0]` |
| YOLO26n-sem | `/255` | `[1, 2, 80, 80]` (stride 8) | `out[:,1] − out[:,0]` |

The pixel-scale conversion in `model_input()` — converts the neutral `/255` loader tensor to each checkpoint's trained scale: ImageNet
norm for SegFormer, `/255` left unchanged for YOLO. (v1 did the reverse -- ImageNet in the
loader, de-normalise YOLO -- verified equivalent to ~6e-8.) Only the pixel *scale* differs;
the cv2 640 resize, interpolation and geometry stay byte-identical across all 5 models.

### About `F.interpolate`

It can't be removed, and it isn't a preprocessing choice — it's structural. **Neither
head emits at 640.** SegFormer's decode head emits 160×160 and `SegformerWrapper`
bilinear-upsamples to 640 *inside its own `forward()`* — that's how it was trained and
scored. YOLO's emits 80×80. The guard applies **the same bilinear upsample to both**: a
no-op for SegFormer, an 8× upsample for YOLO. One operation, both families. The sanity
check prints each head's real shape, so these are observed, not assumed.

What *is* gone: no `cv2.resize` on any model output, no nearest-vs-bilinear mismatch, no
mask ever resized in order to be compared against GT.

### The mask never leaves the GPU

`bruise_mask_640` returns a **CUDA** tensor. The only `.cpu()` is in the accuracy loop,
where numpy metrics genuinely need it — and that loop is not timed.

In [ ]:
# ImageNet mean/std. The loader now emits /255, so model_input() APPLIES this norm for
# SegFormer ('imagenet') and leaves YOLO ('01') at /255 -- see the markdown above.
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


def model_input(entry, x):
    """Convert the neutral /255 loader tensor to the pixel scale THIS checkpoint was trained on.

    per_model_batch convention: the dataloader is model-agnostic (/255), each model rescales
    HERE. v1 did the reverse (ImageNet in the loader, de-normalise YOLO); the two are
    algebraically identical, v2 flips it to match per_model_batch's neutral-loader design.

    'imagenet' (SegFormer): apply ImageNet norm -- its MiT encoder was pretrained that way and
                            SegformerWrapper expects normalised pixel_values.
    '01'       (YOLO):      leave as /255 -- Ultralytics trained on /255; ImageNet-normalised
                            pixels cap it at 0.479 Dice and no threshold recovers it.
    """
    return (x - IMAGENET_MEAN) / IMAGENET_STD if entry['input'] == 'imagenet' else x


def bruise_logit_640(entry, x):
    """[B,3,640,640] normalised GPU tensor -> [B,640,640] bruise logit on the 640 grid.

    Branches on channel count -- a structural fact -- never on a model's name:
      1 channel  (SegFormer): head already emits a bruise logit -> out[:, 0]
      2 channels (YOLO):      2-class softmax == sigmoid(z1-z0) -> out[:,1] - out[:,0]
    """
    out = entry['model'](model_input(entry, x))
    if isinstance(out, (tuple, list)):        # YOLO returns (preds, aux) in eval mode
        out = out[0]
    out = out.float()

    # Same bilinear upsample for both families. No-op for SegFormer (its wrapper already
    # upsampled 160 -> 640 internally); an 8x upsample from 80 -> 640 for YOLO.
    if out.shape[-2:] != (IMG_H, IMG_W):
        out = F.interpolate(out, size=(IMG_H, IMG_W), mode='bilinear', align_corners=False)

    return out[:, 1] - out[:, 0] if out.shape[1] >= 2 else out[:, 0]


def bruise_mask_640(entry, x):
    """[B,3,640,640] GPU tensor -> [B,640,640] bool mask. Stays on the GPU.

    sigmoid(z) >= thr is the same rule as the cut z >= logit(thr) the sweep works in,
    written in probability space because that is the interpretable form.
    """
    if entry['threshold'] is None:
        raise RuntimeError(f"{entry['display']} has no threshold yet — run the val sweep (§10).")
    return torch.sigmoid(bruise_logit_640(entry, x)) >= entry['threshold']


# Sanity check. pred_frac@cut0 is the real guard here: a model fed the wrong pixel scale
# under-fires badly, and that shows up as a predicted-positive fraction far below the GT
# rate -- exactly how the ImageNet-vs-/255 bug was caught (YOLO predicted 0.0118 against
# a GT rate of 0.0473). Numbers far below GT_POS_FRAC mean something is wrong.
GT_POS_FRAC = (GT_640 > 0.5).float().mean().item()
print(f"GT positive fraction: {GT_POS_FRAC:.4f}   (a sane model predicts roughly this)\n")

with torch.inference_mode():
    for m in MODELS.values():
        raw = m['model'](model_input(m, X_TEST[:1]))
        raw = raw[0] if isinstance(raw, (tuple, list)) else raw
        z = bruise_logit_640(m, X_TEST[:8])
        assert z.shape == (8, IMG_H, IMG_W) and z.is_cuda, (z.shape, z.device)
        print(f"{m['display']:26s} in={m['input']:8s} head {str(tuple(raw.shape)):18s} "
              f"-> logit {tuple(z.shape)} | pred_frac@cut0 {(z >= 0).float().mean():.4f}")

---
# 10. Threshold sweep — 1-D, on val, same mechanism for all 5 models

### What is swept

The cut `c` on the raw bruise logit: `mask = (z >= c)`. That is the space the decision
actually lives in. The equivalent probability threshold is `σ(c)`, which is what §11–12
then use — the two are the same rule written two ways.

One forward pass per val image, **cached**, then all cuts scored against the cached
logits. The old sweep re-ran the model for every grid point — 152 × 134 = **20,368**
forward passes per model. This does **134**, then the sweep is pure tensor math.

### How the winner is chosen — *not* `argmax`

The old sweep took `iloc[0]` after sorting by mean_dice. On 134 images its top-8 grid
points spanned **7×10⁻⁴** in mean Dice, with rows 1 and 2 differing by **1.1×10⁻⁵** —
so `iloc[0]` was selecting on noise, not signal.

Instead: compute the standard error of the mean Dice at the peak, take **every cut within
1 SE of the best** as statistically tied, and choose the **median of that tied band**.
That lands in the middle of the plateau rather than on whichever noise spike happened to
top the list, so the choice is stable — and the printed band width tells you honestly how
well-determined the threshold really is.

Fitted on **val (134)**. Never on test.

> A wide tied band is not a bug — it's the BCE-saturation effect being reported instead of
> hidden. A head trained to push logits to ±∞ leaves a large empty region in the middle of
> its logit histogram, and *every* cut inside that gap produces the same mask.

In [ ]:
# Cut grid on the raw logit. Wide enough to cover a saturated head: sigmoid(+-12) is
# ~6e-6 / ~0.999994, far outside the [0.05, 0.95] the old probability-space sweep could
# reach. We check the winner isn't at an edge rather than trusting the range.
CUT_GRID = torch.linspace(-12.0, 12.0, 481, device=DEVICE)    # step 0.05


def val_logits_and_gt(entry):
    """Forward the whole val set once; cache bruise logits + GT on the GPU.

    134 x 640 x 640 floats ~= 219 MB, plus a 55 MB bool GT. Cheap, and it means the
    481-point sweep costs one forward pass over val, not 481 of them.
    """
    zs, gs = [], []
    for x, y, *_ in val_loader:
        zs.append(bruise_logit_640(entry, x.to(DEVICE, non_blocking=True)))
        gs.append((y[:, 0] > 0.5).to(DEVICE, non_blocking=True))
    return torch.cat(zs), torch.cat(gs)


def dice_per_image(z, gt, cut):
    """Per-image Dice at one cut. Matches pipeline.metrics.dice_np, including its
    empty-prediction-AND-empty-GT convention (denominator 0 -> Dice 1.0)."""
    pred = z >= cut
    inter = (pred & gt).sum(dim=(1, 2)).float()
    denom = pred.sum(dim=(1, 2)).float() + gt.sum(dim=(1, 2)).float()
    return torch.where(denom > 0, 2.0 * inter / denom, torch.ones_like(denom))


def sweep(entry):
    """1-D cut sweep on val. Returns the full grid plus the chosen operating point."""
    with torch.inference_mode():
        z, gt = val_logits_and_gt(entry)

        means = torch.empty(len(CUT_GRID), device=DEVICE)
        for i, c in enumerate(CUT_GRID):
            means[i] = dice_per_image(z, gt, c).mean()

        cuts = CUT_GRID.cpu().numpy()
        md = means.cpu().numpy()
        best_i = int(md.argmax())

        # Standard error of the mean Dice at the peak defines the tie band: any cut
        # scoring within 1 SE of the best is not distinguishable from it on 134 images.
        peak = dice_per_image(z, gt, CUT_GRID[best_i])
        se = float(peak.std(unbiased=True) / np.sqrt(peak.numel()))

        del z, gt
        torch.cuda.empty_cache()

    tied = cuts[md >= md[best_i] - se]
    chosen_cut = float(np.median(tied))
    chosen_thr = float(1.0 / (1.0 + np.exp(-chosen_cut)))

    if best_i in (0, len(cuts) - 1):
        print(f"  !! {entry['display']}: optimum sits at the EDGE of the cut grid "
              f"({cuts[best_i]:+.2f}). Widen CUT_GRID — the true optimum may be outside it.")

    grid = pd.DataFrame({'cut': cuts, 'threshold': 1.0 / (1.0 + np.exp(-cuts)), 'mean_dice': md})
    return {'grid': grid, 'cut': chosen_cut, 'threshold': chosen_thr, 'se': se,
            'argmax_cut': float(cuts[best_i]), 'peak_dice': float(md[best_i]),
            'tie_lo': float(tied.min()), 'tie_hi': float(tied.max()), 'n_tied': int(tied.size)}


print(f'Sweeping {len(CUT_GRID)} cuts on {len(val_df)} val images, per model.\n')
sweeps = {}
for run, m in MODELS.items():
    s = sweeps[run] = sweep(m)
    m['threshold'] = s['threshold']       # <-- the operating point everything below uses
    m['cut'] = s['cut']
    print(f"{m['display']:26s} cut={s['cut']:+.3f} -> threshold={s['threshold']:.4f} | "
          f"val peak Dice={s['peak_dice']:.4f}")
    print(f"{'':26s} tied band: {s['n_tied']:3d}/{len(CUT_GRID)} cuts within 1 SE "
          f"({s['se']:.2e}) -> cut in [{s['tie_lo']:+.2f}, {s['tie_hi']:+.2f}]")
    if m['csv_threshold'] is not None:
        print(f"{'':26s} historical CSV equiv threshold: {m['csv_threshold']:.4f} "
              f"(delta {s['threshold']-m['csv_threshold']:+.4f})")

### The sweep curve

The flat top is the point. Where the curve is flat, the threshold is genuinely
underdetermined by 134 val images, and any "best" pulled from that region is a coin
flip between equivalent options. The shaded band is the 1-SE tie region; the dashed
line is the median cut we selected from it.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(MODELS), figsize=(3.6 * len(MODELS), 3.2), sharey=True)
axes = np.atleast_1d(axes)

for ax, (run, m) in zip(axes, MODELS.items()):
    s, g = sweeps[run], sweeps[run]['grid']
    ax.plot(g.cut, g.mean_dice, lw=1.4, color='#1f77b4')

    # The 1-SE tie band: every cut in here is statistically indistinguishable from
    # the peak on 134 images. Its width IS the honest uncertainty on the threshold.
    ax.axvspan(s['tie_lo'], s['tie_hi'], color='orange', alpha=0.18,
               label=f"1-SE tie band ({s['n_tied']} cuts)")
    ax.plot(s['argmax_cut'], s['peak_dice'], 'o', ms=5, color='k',
            label=f"argmax {s['argmax_cut']:+.2f}")
    ax.axvline(s['cut'], ls='--', c='crimson', lw=1.3,
               label=f"chosen {s['cut']:+.2f}")

    ax.set_title(m['display'], fontsize=9)
    ax.set_xlabel('cut on bruise logit')
    ax.set_xlim(-8, 8)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=6.5, loc='lower center')

axes[0].set_ylabel('val mean Dice')
plt.tight_layout()
plt.show()

---
# 11. Benchmark — 640 in → 640 out, on the GPU, over all 185 test images

### ✅ INCLUDED in the timer
- model forward pass
- bilinear upsample of logits to 640
- logit difference (YOLO) / channel select (SegFormer)
- threshold on the raw logit (`z >= cut`) — per_model_batch style, no sigmoid

### ❌ EXCLUDED from the timer
- disk read / decode
- resize to 640 (done once when the 640 cache is built, §6) + `/255` scaling (dataloader)
- sigmoid — a hard mask needs only `z >= cut`; sigmoid is monotonic so it changes no pixel
- host→device copy (done once, §7 — tensors are already resident)
- **any device→host copy** — the mask stays on the GPU
- model loading and warmup

### Two things that make GPU timing correct

* **`torch.cuda.synchronize()`** — the GPU is *asynchronous*. Without it we'd time how
  long it takes to *queue* the work (microseconds), not to *do* it (milliseconds).
* **Warmup** — the first calls pay one-off costs (cuDNN autotuning, allocator growth).

All 185 images are timed, `N_REPEATS` times each, so the spread is real across-image
variation rather than one image's noise. Median is the typical case; p95 is the "how bad
does it get" number a responsive UI actually cares about.

In [ ]:
N_WARMUP, N_REPEATS = 20, 3

def benchmark(entry):
    """[per_model_batch step 3] Time the raw-logit path on every staged test image, N_REPEATS each.

    Matches per_model_batch's benchmark_speed: forward -> (upsample+channel-select via
    bruise_logit_640) -> threshold on the LOGIT (z >= cut). NO sigmoid -- it is monotonic, so
    z >= cut yields the identical mask as sigmoid(z) >= sigmoid(cut), and skipping it is one
    fewer timed op. cut is the val-fitted logit cut from the sweep (§10).
    """
    n   = X_TEST.shape[0]
    cut = entry['cut']
    with torch.inference_mode():
        for i in range(N_WARMUP):                       # steady state before we measure
            z = bruise_logit_640(entry, X_TEST[i % n:i % n + 1]); _ = z >= cut
        torch.cuda.synchronize(DEVICE)

        # Everything already resident (X_TEST ~0.9 GB + all 5 models' weights) is not THIS
        # model's inference cost, so subtract the baseline and report only the transient
        # activation one forward pass adds.
        torch.cuda.reset_peak_memory_stats(DEVICE)
        baseline_b = torch.cuda.memory_allocated(DEVICE)

        lat = []
        for _ in range(N_REPEATS):
            for i in range(n):
                x = X_TEST[i:i + 1]                     # already on GPU, already 640, already /255
                torch.cuda.synchronize(DEVICE)          # GPU idle before we start
                t0 = time.perf_counter()
                z = bruise_logit_640(entry, x)          # forward + upsample + channel select
                _ = z >= cut                            # threshold on the raw logit (no sigmoid)
                torch.cuda.synchronize(DEVICE)          # wait for it to actually finish
                lat.append((time.perf_counter() - t0) * 1000.0)

    lat = np.asarray(lat)
    return {'mean_ms': lat.mean(), 'median_ms': np.median(lat),
            'p95_ms': np.percentile(lat, 95), 'std_ms': lat.std(),
            'fps': 1000.0 / lat.mean(), 'n_timed': int(lat.size),
            'activation_mb': (torch.cuda.max_memory_allocated(DEVICE) - baseline_b) / 1024**2}

print(f'Timing {len(test_df)} images x {N_REPEATS} repeats = '
      f'{len(test_df)*N_REPEATS} timed calls per model, after {N_WARMUP} warmup.\n')

bench = {}
for run, m in MODELS.items():
    bench[run] = b = benchmark(m)
    print(f"{m['display']:26s} {b['median_ms']:6.2f} ms (median) | p95 {b['p95_ms']:6.2f} | "
          f"{b['fps']:6.1f} FPS | {b['n_timed']} calls | +{b['activation_mb']:5.1f} MB act.")

---
# 12. Accuracy — all 5 models on all 185 test images at 640

Same `bruise_mask_640` the benchmark just timed, same tensors, same val-fitted
thresholds. The only additions are the GT comparison and the `.cpu()` numpy metrics
need — which is exactly why they live here and not in the timed loop.

Prediction and GT are both already on the dataloader's 640 grid, so **nothing is resized
to make the comparison**.

**Metrics.** Dice/IoU measure overlap. **Complete-miss rate** is the fraction of images
with a bruise where the model predicts *nothing* — reported separately because a blank
mask is a missed injury, far worse than a loose outline.

In [ ]:
per_image = {}
with torch.inference_mode():
    for run, m in MODELS.items():
        rows = []
        for i in range(X_TEST.shape[0]):
            pred = bruise_mask_640(m, X_TEST[i:i+1])[0].to(torch.uint8).cpu().numpy()
            gt   = (GT_640[i, 0].numpy() > 0.5).astype('uint8')
            rows.append(compute_image_row(pred, gt, STEMS[i]))
            if (i + 1) % 50 == 0:
                print(f"  [{m['display']}] {i+1}/{X_TEST.shape[0]}")

        df = pd.DataFrame(rows)
        df['complete_miss'] = (df.pred_positive_pixels == 0) & (df.gt_positive_pixels > 0)
        per_image[run] = df
        print(f"{m['display']:26s} median Dice={df.dice.median():.3f}  mean={df.dice.mean():.3f}  "
              f"miss={df.complete_miss.mean()*100:.2f}%")

## Final results

One preprocessing recipe, one inference path, one sweep mechanism, one timing scope.
Thresholds fitted on val (134), everything reported here measured on test (185).

Read `complete_miss_%` next to `median_Dice` — the best-Dice model is not necessarily
the one you want in the field.

In [ ]:
results = pd.DataFrame([{
    'model': m['display'],
    'params_M': round(m['params_m'], 2),
    # --- fitted on val ---
    'cut': round(m['cut'], 3),
    'threshold': round(m['threshold'], 4),
    'tie_band': f"[{sweeps[run]['tie_lo']:+.2f}, {sweeps[run]['tie_hi']:+.2f}]",
    'val_peak_Dice': round(sweeps[run]['peak_dice'], 4),
    # --- measured on test ---
    'median_Dice': round(per_image[run].dice.median(), 4),
    'mean_Dice': round(per_image[run].dice.mean(), 4),
    'median_IoU': round(per_image[run].iou.median(), 4),
    'complete_miss_%': round(per_image[run].complete_miss.mean() * 100, 2),
    'median_ms': round(bench[run]['median_ms'], 2),
    'p95_ms': round(bench[run]['p95_ms'], 2),
    'FPS': round(bench[run]['fps'], 1),
    'activation_MB': round(bench[run]['activation_mb'], 1),
} for run, m in MODELS.items()])
results

## Save everything to Drive

Per-image test CSVs, the full val sweep grid per model (so the flat top is auditable,
not just the winner), the results table, and metadata recording exactly what was fitted
where and what the stopwatch included.

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
OUT = f'{LOCAL_DIR}/inference_demo_v2'
os.makedirs(OUT, exist_ok=True)

for run, df in per_image.items():
    df.to_csv(f'{OUT}/per_image_test_{run}.csv', index=False)
for run, s in sweeps.items():
    s['grid'].to_csv(f'{OUT}/val_sweep_{run}.csv', index=False)
results.to_csv(f'{OUT}/results_accuracy_and_speed.csv', index=False)

json.dump({
  'gpu': torch.cuda.get_device_name(0),
  'n_val_images': int(len(val_df)), 'n_test_images': int(len(test_df)),
  'scoring_grid': '640x640',
  'preprocessing': ('640-cache /255 dataloader (native images cv2-resized to 640 ONCE into a PNG cache) for ALL 5 models -- one disk read, one '
                    'cv2/albumentations A.Resize to 640x640 (image bilinear, mask nearest). '
                    'Geometry is byte-identical across all 5; no letterbox. PIXEL SCALE is '
                    'per-model and must be: SegFormer is normalised to ImageNet inside model_input() '
                    'and YOLO is kept at the loader's /255 (v2 neutral-loader design, == v1 to ~6e-8) '
                    'because Ultralytics trained it on /255. MEASURED: feeding YOLO '
                    'ImageNet-normalised pixels caps it at 0.479 Dice at its OWN optimal '
                    'threshold vs 0.794 on /255 -- it under-fires 4x (1.2% predicted '
                    'positive vs 4.7% GT) and no threshold recovers it.'),
  'yolo_input_scale_measurement': {
      'imagenet_norm': {'dice_at_argmax': 0.3092, 'best_dice_any_threshold': 0.4791},
      'unit_255':      {'dice_at_argmax': 0.7507, 'best_dice_any_threshold': 0.7940},
      'note': 'measured on 30 val images, yolo_sem_direct, identical cv2 stretch resize',
  },
  'inference_path': ('Raw PyTorch nn.Module for all 5 models. Logits bilinear-upsampled to '
                     '640 (identical op both families), reduced to a single bruise logit '
                     '(1ch: as-is; 2ch: z1-z0 == 2-class softmax), then sigmoid(z) >= thr. '
                     'Ultralytics .predict() is never called.'),
  'threshold_sweep': {
      'fitted_on': f'val split, {len(val_df)} images, 0 image/subject overlap with test',
      'parameterisation': ('1-D cut c on the raw bruise logit (mask = z >= c); reported '
                           'threshold is sigmoid(c). Temperature is NOT swept: for a hard '
                           'mask sigmoid(z/T) >= thr <=> z >= T*logit(thr), so (T, thr) is a '
                           'redundant 2-D parameterisation of this same 1-D cut.'),
      'grid': f'{len(CUT_GRID)} cuts, linspace(-12, 12), step 0.05',
      'forward_passes_per_model': int(len(val_df)),
      'selection_rule': ('median of all cuts within 1 standard error of the peak mean Dice '
                         '-- NOT argmax, which on 134 images selects on noise (the old '
                         'grid\'s top-8 spanned 7e-4 in mean Dice).'),
  },
  'operating_points': {run: {'cut': m['cut'], 'threshold': m['threshold'],
                             'input_scale': m['input'],
                             'tie_band_cut': [sweeps[run]['tie_lo'], sweeps[run]['tie_hi']],
                             'n_tied_cuts': sweeps[run]['n_tied'],
                             'val_peak_dice': sweeps[run]['peak_dice'],
                             'historical_csv_equivalent': m['csv_threshold']}
                       for run, m in MODELS.items()},
  'benchmark': {
      'scope': '640 GPU tensor in -> 640 GPU mask out',
      'included': ['forward pass', 'bilinear upsample of logits to 640',
                   'logit difference / channel select', 'threshold on raw logit (no sigmoid)'],
      'excluded': ['disk read/decode', 'resize to 640', 'ImageNet normalisation',
                   'host->device copy (tensors pre-staged on GPU)',
                   'device->host copy (mask never leaves the GPU)',
                   'model loading', 'warmup'],
      'protocol': f'{N_WARMUP} warmup, then all {len(test_df)} test images x {N_REPEATS} '
                  f'repeats = {len(test_df)*N_REPEATS} timed calls per model, '
                  'cuda.synchronize() around each call, batch_size=1',
  },
  'historical_csv_warning': ('The pre-existing threshold_search.csv files could NOT be '
                            'reproduced: at their own best cut (-1.735) this code measures '
                            '0.4330 val mean Dice on the same 134 images where the CSV '
                            'claims 0.7375. Those CSVs are read here for reference only '
                            '(historical_csv_equivalent) and no operating point depends on '
                            'them -- every threshold below is swept fresh in-notebook.'),
  'licensing_note': ('Inference runs on the raw nn.Module, so Ultralytics\' inference/'
                     'postprocessing stack is not in the runtime path. Loading best.pt '
                     'still unpickles Ultralytics classes, so `import ultralytics` remains '
                     'a dependency. TorchScript/ONNX export removes the technical '
                     'dependency; the legal question is separate and not settled here.'),
}, open(f'{OUT}/metadata.json', 'w'), indent=2)

dest = f'{DRIVE_DIR}/inference_demo_v2_{stamp}'
shutil.copytree(OUT, dest)
print('Saved to:', dest)
!ls -la {dest}